In [10]:
import pathlib
import cogent3

primates_100 = pathlib.Path('~/source/ensembl/primates100')
thesis_data = pathlib.Path('~/source/ensembl/thesis_data')

In [11]:
@cogent3.app.composable.define_app
def rename(align: cogent3.app.typing.AlignedSeqsType)->cogent3.app.typing.AlignedSeqsType:
    return align.rename_seqs(lambda x: x.split(':')[0])

loader = cogent3.get_app('load_aligned', moltype='dna')

renamer = rename()
select_seqs = cogent3.get_app('take_named_seqs','macaca_mulatta','homo_sapiens','gorilla_gorilla','pan_troglodytes')

in_dstore = cogent3.open_data_store(primates_100, suffix='fa') 
out_dstore = cogent3.open_data_store(thesis_data / 'ensembl_alignments', suffix='fa', mode='w') 

writer = cogent3.get_app('write_seqs', data_store = out_dstore)

app = loader + renamer + select_seqs + writer 
app.apply_to(in_dstore,show_progress=True)

   0%|          |00:00<?

DataStoreDirectory(source=/home/richard/source/ensembl/thesis_data/ensembl_alignments, mode=Mode.w, suffix=fa, limit=None, verbose=False)

In [12]:
app.disconnect()
in_dstore = cogent3.open_data_store(thesis_data/'ensembl_alignments', suffix='fa') 
out_dstore = cogent3.open_data_store(thesis_data/'ensembl_alignments'/'human_gorilla', suffix='fa', mode='w') 
loader = cogent3.get_app('load_aligned', moltype='dna')
writer = cogent3.get_app('write_seqs', data_store = out_dstore)
select_seqs = cogent3.get_app('take_named_seqs','homo_sapiens','gorilla_gorilla')
omit_gaps = cogent3.get_app('omit_gap_pos', moltype="dna")
app = loader + select_seqs + omit_gaps + writer 
app.apply_to(in_dstore, show_progress=True)

   0%|          |00:00<?

DataStoreDirectory(source=/home/richard/source/ensembl/thesis_data/ensembl_alignments/human_gorilla, mode=Mode.w, suffix=fa, limit=None, verbose=False)

In [13]:
import cogent3

@cogent3.app.composable.define_app
def c3_pairwise(seqs: cogent3.app.typing.SeqsCollectionType) -> cogent3.app.typing.AlignedSeqsType:
    seqs = seqs.degap()
    # Perform global pairwise alignment
    score_matrix = cogent3.align.align.make_dna_scoring_dict(match=5, transition=-2, transversion=-4)
    gap_penalty = 4
    gap_extend = 1 
    alignment = cogent3.align.global_pairwise(seqs.seqs[0], seqs.seqs[1], score_matrix, gap_penalty, gap_extend)
    alignment.info.source = seqs.info.source
    return alignment

In [14]:
app.disconnect()
in_dstore = cogent3.open_data_store(thesis_data/'ensembl_alignments'/'human_gorilla', suffix='fa') 
out_dstore = cogent3.open_data_store(thesis_data/'cogent3_alignments'/'human_gorilla', suffix='fa', mode='w') 
loader = cogent3.get_app('load_aligned', moltype='dna')
writer = cogent3.get_app('write_seqs', data_store = out_dstore)
app = loader + c3_pairwise() + writer 
app.apply_to(in_dstore, show_progress=True)

   0%|          |00:00<?

DataStoreDirectory(source=/home/richard/source/ensembl/thesis_data/cogent3_alignments/human_gorilla, mode=Mode.w, suffix=fa, limit=None, verbose=False)